## THIS SECTION IS TAKING REGISTRATION DATA FROM THE 2024.PDF FILES

Step 1: Import libraries with tabula first

In [2]:
import tabula
import pandas as pd

Step 2: Read the .pdf

In [4]:
# cell 2: read tables from PDF (returns a list of DataFrames)
pdf_path = "C:/Users/mutch/Democratic_Voter_Shift/data/raw/2024_voterreg.pdf"                    
dfs = tabula.read_pdf(pdf_path, 
    pages="all",
    guess=False,
    stream=True,
    multiple_tables=True
)

Step 3: View what was read

In [5]:
# show how many tables were found
len(dfs)

3

In [6]:
# view the first table
dfs[0]

,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Commonwealth of Pennsylvania,NaN,Department of State
2,2024 Voter Registration Statistics,NaN,Bureau of Elections
3,"November 5, 2024 Election",NaN,NaN
4,NaN,NaN,NaN
5,County Democratic Republican Libertarian,NaN,Other Parties All Parties
6,"Adams 19,474 43,914",449,"12,171 76,008"
7,"Allegheny 526,268 274,157","5,088","147,030 952,543"
8,"Armstrong 11,418 27,984",237,"5,032 44,671"
9,"Beaver 48,284 52,318",661,"16,185 117,448"


In [7]:
# prettier display in notebooks
display(dfs[0])

,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Commonwealth of Pennsylvania,NaN,Department of State
2,2024 Voter Registration Statistics,NaN,Bureau of Elections
3,"November 5, 2024 Election",NaN,NaN
4,NaN,NaN,NaN
5,County Democratic Republican Libertarian,NaN,Other Parties All Parties
6,"Adams 19,474 43,914",449,"12,171 76,008"
7,"Allegheny 526,268 274,157","5,088","147,030 952,543"
8,"Armstrong 11,418 27,984",237,"5,032 44,671"
9,"Beaver 48,284 52,318",661,"16,185 117,448"


Step 4: Combine all tables into 1 table in a .CSV

In [8]:
# combine all extracted tables into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# view the result
display(combined_df)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Commonwealth of Pennsylvania,NaN,Department of State,NaN,NaN
2,2024 Voter Registration Statistics,NaN,Bureau of Elections,NaN,NaN
3,"November 5, 2024 Election",NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
83,NaN,NaN,NaN,NaN,NaN
84,NaN,NaN,NaN,NaN,NaN
85,NaN,NaN,NaN,NaN,NaN
86,NaN,Bureau of Elections,NaN,NaN,NaN


In [12]:
combined_df.head(50)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Commonwealth of Pennsylvania,NaN,Department of State,NaN,NaN
2,2024 Voter Registration Statistics,NaN,Bureau of Elections,NaN,NaN
3,"November 5, 2024 Election",NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,County Democratic Republican Libertarian,NaN,Other Parties All Parties,NaN,NaN
6,"Adams 19,474 43,914",449,"12,171 76,008",NaN,NaN
7,"Allegheny 526,268 274,157","5,088","147,030 952,543",NaN,NaN
8,"Armstrong 11,418 27,984",237,"5,032 44,671",NaN,NaN
9,"Beaver 48,284 52,318",661,"16,185 117,448",NaN,NaN


In [13]:
# save to CSV
combined_df.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024.csv", index=False)

Step : Clean the table

In [14]:
input_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024.csv"
output_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_a.csv"

df = pd.read_csv(input_path)

rows_to_drop = list(range(0, 5)) + list(range(36, 39)) + list(range(73, 76)) + list(range(80, 88))
df = df.drop(index=rows_to_drop).reset_index(drop=True)

df.to_csv(output_path, index=False)

df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,County Democratic Republican Libertarian,NaN,Other Parties All Parties,NaN,NaN
1,"Adams 19,474 43,914",449,"12,171 76,008",NaN,NaN
2,"Allegheny 526,268 274,157","5,088","147,030 952,543",NaN,NaN
3,"Armstrong 11,418 27,984",237,"5,032 44,671",NaN,NaN
4,"Beaver 48,284 52,318",661,"16,185 117,448",NaN,NaN


This file is a mess.  I want to first eliminate the first row, which is index 0

In [15]:
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_a.csv"

df = pd.read_csv(file_path)

# remove row 1 from the file
# pandas uses zero-based indexing, so row 1 is index 0
df = df.drop(index=0).reset_index(drop=True)

# overwrite the existing file
df.to_csv(file_path, index=False)

In [20]:
import re

file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_a.csv"
outfile_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_b.csv."

df = pd.read_csv(file_path, header=None)

# helper to split numeric strings like 12,345 into 12345
def clean_num(x):
    if pd.isna(x):
        return None
    x = str(x).strip().replace(",", "")
    return x if x == "" else x

# build corrected rows
fixed_rows = []

# rows 1-30 in the file -> pandas index 0-29
for i in range(0, 31):
    row = df.iloc[i].tolist()
    c1 = str(row[0]) if pd.notna(row[0]) else ""
    c2 = clean_num(row[1])
    c3 = clean_num(row[2])

    # split first column into county, democratic, republican
    # adjust if your county names contain spaces and the numbers are always at the end
    m = re.match(r"^(.*?)(\d[\d,]*)\s+(\d[\d,]*)$", c1.strip())
    if m:
        county = m.group(1).strip()
        democratic = m.group(2).replace(",", "")
        republican = m.group(3).replace(",", "")
    else:
        parts = c1.split()
        county = " ".join(parts[:-2])
        democratic = parts[-2].replace(",", "")
        republican = parts[-1].replace(",", "")

    fixed_rows.append([
        county,
        democratic,
        republican,
        c2,
        c3,
        row[3] if len(row) > 3 else None
    ])

# rows 31-68 in the file -> pandas index 30-67
for i in range(32, 69):
    row = df.iloc[i].tolist()
    c1 = str(row[0]) if pd.notna(row[0]) else ""

    # split first column into county and democratic
    m = re.match(r"^(.*?)(\d[\d,]*)$", c1.strip())
    if m:
        county = m.group(1).strip()
        democratic = m.group(2).replace(",", "")
    else:
        parts = c1.split()
        county = " ".join(parts[:-1])
        democratic = parts[-1].replace(",", "")

    republican = row[1] if len(row) > 1 else None
    libertarian = row[2] if len(row) > 2 else None
    other_parties = row[3] if len(row) > 3 else None
    all_parties = row[4] if len(row) > 4 else None

    fixed_rows.append([
        county,
        democratic,
        republican,
        libertarian,
        other_parties,
        all_parties
    ])

# create corrected dataframe
corrected_df = pd.DataFrame(
    fixed_rows,
    columns=[
        "county",
        "democratic",
        "republican",
        "libertarian",
        "other_parties",
        "all_parties"
    ]
)

# save as
corrected_df.to_csv(outfile_path, index=False)

corrected_df.head()

,county,democratic,republican,libertarian,other_parties,all_parties
0,,Unnamed:,0,Unnamed: 1,Unnamed: 2,Unnamed: 3
1,Adams,19474,43914,449,12171 76008,NaN
2,Allegheny,526268,274157,5088,147030 952543,NaN
3,Armstrong,11418,27984,237,5032 44671,NaN
4,Beaver,48284,52318,661,16185 117448,NaN


drop row 1

In [22]:
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_b.csv"

df = pd.read_csv(file_path)

# drop row 1 (the first data row; pandas index 0)
df = df.drop(index=0).reset_index(drop=True)

df.to_csv(file_path, index=False)

df.head()

,county,democratic,republican,libertarian,other_parties,all_parties
0,Adams,19474,43914,449,12171 76008,NaN
1,Allegheny,526268,274157,5088,147030 952543,NaN
2,Armstrong,11418,27984,237,5032 44671,NaN
3,Beaver,48284,52318,661,16185 117448,NaN
4,Bedford,5671,24907,120,3103 33801,NaN


Fix columns 5-6, rows 1-30

In [23]:
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_b.csv"

df = pd.read_csv(file_path)

# rows 1-30 -> pandas index 0:30
# split column 5 (index 4) into other_parties and all_parties
split_vals = df.iloc[0:30, 4].astype(str).str.strip().str.split(r"\s+", n=1, expand=True)

df.loc[0:29, "other_parties"] = split_vals[0].values
df.loc[0:29, "all_parties"] = split_vals[1].values

# drop the original extra column if needed
# if column 6 is now redundant or you want to rebuild the row structure, adjust here
df.to_csv(file_path, index=False)

df.head(30)

,county,democratic,republican,libertarian,other_parties,all_parties
0,Adams,19474,43914,449,12171,76008
1,Allegheny,526268,274157,5088,147030,952543
2,Armstrong,11418,27984,237,5032,44671
3,Beaver,48284,52318,661,16185,117448
4,Bedford,5671,24907,120,3103,33801
5,Berks,109012,118848,1729,47241,276830
6,Blair,20009,50687,377,10228,81301
7,Bradford,8277,25720,225,5090,39312
8,Bucks,200835,206709,2699,83627,493870
9,Butler,40520,83902,892,20406,145720


Finally, remove row 67, and add a column for the year 2024

In [24]:
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_b.csv"

df = pd.read_csv(file_path)

# remove row 67
# if you mean the 67th data row, pandas index is 66
df = df.drop(index=66).reset_index(drop=True)

# insert year after county
if "county" in df.columns:
    county_pos = df.columns.get_loc("county")
    df.insert(county_pos + 1, "year", 2024)
else:
    raise KeyError("Expected a 'county' column, but it was not found.")

# overwrite the file
df.to_csv(file_path, index=False)

df.head()

,county,year,democratic,republican,libertarian,other_parties,all_parties
0,Adams,2024,19474,43914,449,12171,76008
1,Allegheny,2024,526268,274157,5088,147030,952543
2,Armstrong,2024,11418,27984,237,5032,44671
3,Beaver,2024,48284,52318,661,16185,117448
4,Bedford,2024,5671,24907,120,3103,33801


Finally, add Huntington county and order alphabetically by county

In [25]:
file_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/combined_2024_b.csv"
outfile_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/VoterReg2024_cleaned.csv"

df = pd.read_csv(file_path)

new_row = {
    "county": "Huntington",
    "year": 2024,
    "democratic": 6492,
    "republican": 19202,
    "libertarian": 113,
    "other_parties": 2921,
    "all_parties": 28718
}

df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

df = df.sort_values(by="county").reset_index(drop=True)

df.to_csv(outfile_path, index=False)

df.head()

,county,year,democratic,republican,libertarian,other_parties,all_parties
0,Adams,2024,19474,43914,449,12171,76008
1,Allegheny,2024,526268,274157,5088,147030,952543
2,Armstrong,2024,11418,27984,237,5032,44671
3,Beaver,2024,48284,52318,661,16185,117448
4,Bedford,2024,5671,24907,120,3103,33801


## SUMMARY: VoterReg2024_cleaned is ready for use